In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import re


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import re
from fake_useragent import UserAgent

class YandexRealtyParser:
    def __init__(self, headless=True):
        self.options = Options()
        if headless:
            self.options.add_argument('--headless=new')
        
        # === АНТИ-ДЕТЕКЦИЯ ===
        self.options.add_argument('--no-sandbox')
        self.options.add_argument('--disable-dev-shm-usage')
        self.options.add_argument('--disable-blink-features=AutomationControlled')
        self.options.add_argument('--disable-gpu')
        self.options.add_argument('--window-size=1920,1080')
        self.options.add_experimental_option('excludeSwitches', ['enable-automation'])
        self.options.add_experimental_option('useAutomationExtension', False)
        
        # Случайный User-Agent
        ua = UserAgent()
        self.options.add_argument(f'user-agent={ua.chrome}')
        self.options.add_argument('--lang=ru-RU,ru;q=0.9')
        
        self.driver = None
        self.data = []
    
    def start(self):
        """Запуск браузера с обходом детекции"""
        self.driver = webdriver.Chrome(service=Service(), options=self.options)
        
        # Скрипт для удаления признаков автоматизации
        self.driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
            'source': '''
                Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
                Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]});
                Object.defineProperty(navigator, 'languages', {get: () => ['ru-RU', 'ru', 'en']});
                window.chrome = {runtime: {}};
            '''
        })
        print("✅ Chrome запущен с анти-детекцией")
    
    def close(self):
        if self.driver:
            self.driver.quit()
            print("✅ Chrome закрыт")
    
    def is_captcha_page(self, html):
        """Проверка на страницу с капчей"""
        captcha_indicators = [
            'smart-captcha', 'checkbox-captcha', 'captcha_smart',
            'подтвердите', 'робот', 'Я не робот', 'checkcaptcha',
            'access denied', 'blocked', '403', '429'
        ]
        html_lower = html.lower()
        return any(ind in html_lower for ind in captcha_indicators)
    
    def get_rendered_html(self, url, max_retries=3):
        """Получение рендеренного HTML с обработкой капчи"""
        for attempt in range(max_retries):
            try:
                print(f"   🔄 Попытка {attempt + 1}/{max_retries}...")
                self.driver.get(url)
                
                # Ждём либо карточки объявления, либо капчу
                try:
                    WebDriverWait(self.driver, 20).until(
                        EC.presence_of_element_located((
                            By.CSS_SELECTOR, 
                            "[data-test='OffersSerpItem'], .CheckboxCaptcha, .OffersSerpItem__title"
                        ))
                    )
                except TimeoutException:
                    print("   ⚠️ Таймаут ожидания контента")
                
                # Прокрутка для подгрузки ленивых изображений
                for _ in range(2):
                    self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    time.sleep(random.uniform(1, 2))
                
                html = self.driver.page_source
                
                # Проверка на капчу
                if self.is_captcha_page(html):
                    print("   ⚠️ Обнаружена CAPTCHA!")
                    if attempt < max_retries - 1:
                        wait_time = random.uniform(15, 30)
                        print(f"   ⏳ Ждём {wait_time:.1f} сек перед повторной попыткой...")
                        time.sleep(wait_time)
                        self.driver.refresh()
                        continue
                    else:
                        print("   ❌ Не удалось обойти капчу")
                        return None
                
                return html
                
            except Exception as e:
                print(f"   ❌ Ошибка: {e}")
                if attempt < max_retries - 1:
                    time.sleep(random.uniform(5, 10))
                    continue
                return None
        
        return None
    
    def clean_text(self, text):
        """Очистка текста от лишних пробелов и символов"""
        if not text:
            return 'N/A'
        return ' '.join(text.split()).replace('\xa0', ' ').strip()
    
    def parse_price_to_int(self, price_text):
        """Конвертация цены в число"""
        if not price_text or price_text == 'N/A':
            return None
        try:
            # Берём первое числовое значение (актуальная цена, если есть скидка)
            match = re.search(r'([\d\s]+)\s*₽', str(price_text))
            if match:
                return int(re.sub(r'[^\d]', '', match.group(1)))
        except:
            pass
        return None
    
    def extract_images(self, item_soup):
        """Извлечение изображений из карточки"""
        images = {
            'main_image': 'N/A',
            'image_urls': [],
            'photo_count': 0
        }
        
        # Ищем галерею
        gallery = item_soup.find('div', {'data-test': 'SnippetGallery'})
        if not gallery:
            return images
        
        # Главное изображение
        main_img = gallery.find('img', class_=lambda x: x and 'Gallery__activeImg' in x)
        if main_img and main_img.get('src'):
            src = main_img.get('src')
            images['main_image'] = f"https:{src}" if src.startswith('//') else src
        
        # Все изображения
        img_tags = gallery.find_all('img', src=lambda x: x and 'realty-offers' in x)
        seen = set()
        for img in img_tags:
            src = img.get('src')
            if src and src not in seen:
                seen.add(src)
                full_url = f"https:{src}" if src.startswith('//') else src
                images['image_urls'].append(full_url)
        
        # Подсчёт фото (из индикатора "+25")
        over_limit = gallery.find('li', class_=lambda x: x and 'BulletIndicator__overLimit' in x)
        if over_limit:
            match = re.search(r'\+\s*(\d+)', over_limit.get_text())
            base_count = len(gallery.find_all('li', class_='BulletIndicator__bullet'))
            extra = int(match.group(1)) if match else 0
            images['photo_count'] = base_count + extra
        else:
            images['photo_count'] = len(images['image_urls'])
        
        return images
    
    def parse_offer(self, item_html):
        """Парсинг одного объявления из HTML"""
        soup = BeautifulSoup(item_html, 'html.parser')
        offer = {}
        
        # 🔗 Ссылка и ID
        link = soup.find('a', href=lambda x: x and '/offer/' in x)
        if link:
            href = link.get('href', '')
            offer['url'] = f"https://realty.yandex.ru{href}" if href.startswith('/') else href
            offer['offer_id'] = href.strip('/').split('/')[-2] if href else 'N/A'
        else:
            offer['url'] = offer['offer_id'] = 'N/A'
        
        # 📐 Заголовок: площадь, комнаты, этаж
        title_el = soup.find('span', class_=lambda x: x and 'OffersSerpItemTitle__title' in x)
        if title_el:
            title = self.clean_text(title_el.get_text())
            offer['title'] = title
            
            # Площадь
            area_match = re.search(r'([\d,]+)\s*м²', title)
            offer['area'] = area_match.group(1).replace(',', '.') if area_match else 'N/A'
            
            # Комнаты
            if 'студия' in title.lower() or 'квартира-студия' in title.lower():
                offer['rooms'] = 'студия'
            else:
                rooms_match = re.search(r'(\d+)-комнатная', title)
                offer['rooms'] = rooms_match.group(1) if rooms_match else 'N/A'
            
            # Этаж
            floor_match = re.search(r'(\d+)\s*этаж\s*из\s*(\d+)', title)
            if floor_match:
                offer['floor'] = f"{floor_match.group(1)}/{floor_match.group(2)}"
            else:
                offer['floor'] = 'N/A'
        else:
            offer.update({'title': 'N/A', 'area': 'N/A', 'rooms': 'N/A', 'floor': 'N/A'})
        
        # 💰 Цена — ИСПРАВЛЕННЫЙ ПАРСИНГ
        price_container = soup.find('div', class_=lambda x: x and 'OfferPriceLabel__priceWithTrend' in x)
        if price_container:
            # Ищем span с цифрами цены
            price_span = None
            for span in price_container.find_all('span'):
                text = span.get_text(strip=True)
                # Проверяем, что это число с ₽
                if text and re.match(r'^[\d\s,]+$', text.replace('₽', '').replace('–', '').strip()):
                    price_span = span
                    break
            
            if price_span:
                raw_price = price_span.get_text(strip=True)
                offer['price'] = self.clean_text(raw_price)
                offer['price_numeric'] = self.parse_price_to_int(offer['price'])
            else:
                # Резерв: берём весь текст и ищем число
                raw = price_container.get_text()
                match = re.search(r'([\d\s,]+)\s*₽', raw)
                if match:
                    offer['price'] = match.group(0).strip()
                    offer['price_numeric'] = self.parse_price_to_int(offer['price'])
                else:
                    offer['price'] = 'N/A'
                    offer['price_numeric'] = None
            
            # Старая цена (если есть скидка)
            old_price_el = price_container.find('span', class_=lambda x: x and 'OfferPriceLabel__oldPriceBase' in x)
            if old_price_el:
                offer['old_price'] = self.clean_text(old_price_el.get_text())
            else:
                offer['old_price'] = 'N/A'
        else:
            offer.update({'price': 'N/A', 'price_numeric': None, 'old_price': 'N/A'})
        
        # 💰 Цена за м²
        ppsm = soup.find('div', class_=lambda x: x and 'OffersSerpItem__price-detail' in x)
        offer['price_per_m2'] = self.clean_text(ppsm.get_text()) if ppsm else 'N/A'
        
        # 🚇 Метро
        metro = soup.find('span', class_=lambda x: x and 'MetroStation__title' in x)
        offer['metro'] = self.clean_text(metro.get_text()) if metro else 'N/A'
        
        # ⏱️ Время до метро
        metro_time = soup.find('span', class_=lambda x: x and 'MetroWithTime__distance' in x)
        offer['metro_time'] = self.clean_text(metro_time.get_text()) if metro_time else 'N/A'
        
        # 🏠 Адрес / ЖК
        address = soup.find('div', class_=lambda x: x and 'OfferSnippetLocation__address' in x)
        offer['address'] = self.clean_text(address.get_text()) if address else 'N/A'
        
        # 📝 Описание
        desc = soup.find('p', class_=lambda x: x and 'OffersSerpItem__description' in x)
        offer['description'] = self.clean_text(desc.get_text()) if desc else 'N/A'
        
        # 📅 Дата публикации
        date = soup.find('div', class_=lambda x: x and 'OffersSerpItem__publish-date' in x)
        if not date:
            date = soup.find('div', class_=lambda x: x and 'OffersSerpItem__publish-date__hide' in x)
        offer['publish_date'] = self.clean_text(date.get_text()) if date else 'N/A'
        
        # 🏢 Автор/застройщик
        author = soup.find('div', class_=lambda x: x and 'OffersSerpAuthor__name' in x)
        offer['author'] = self.clean_text(author.get_text()) if author else 'N/A'
        
        # 🏷️ Бейджи
        badges = [self.clean_text(b.get_text()) for b in soup.find_all('div', class_=lambda x: x and 'Badge__badgeText' in x)]
        offer['badges'] = '; '.join(badges) if badges else 'N/A'
        
        # 🖼️ Изображения
        img_data = self.extract_images(soup)
        offer['main_image'] = img_data['main_image']
        offer['photo_count'] = img_data['photo_count']
        offer['image_urls'] = '; '.join(img_data['image_urls'][:5])  # Первые 5 для компактности
        
        return offer
    
    def parse_page(self, html):
        """Парсинг страницы поиска"""
        soup = BeautifulSoup(html, 'html.parser')
        items = []
        
        # Ищем карточки по data-атрибуту (самый надёжный селектор)
        offers = soup.find_all('li', {'data-test': 'OffersSerpItem'})
        print(f"   📦 Найдено карточек: {len(offers)}")
        
        for offer_el in offers:
            # Пропускаем рекламные блоки
            if offer_el.get('class') and any('ad' in c.lower() for c in offer_el.get('class', [])):
                continue
            
            parsed = self.parse_offer(str(offer_el))
            if parsed and parsed.get('url') != 'N/A':
                items.append(parsed)
        
        return items
    
    def parse_multiple_pages(self, base_url, pages=3):
        """Парсинг нескольких страниц"""
        print(f"🚀 Начинаем парсинг {pages} страниц...")
        print(f"📍 URL: {base_url}\n")
        
        for page in range(1, pages + 1):
            # Формируем URL с пагинацией
            if page == 1:
                url = base_url
            else:
                if '?page=' in base_url:
                    url = re.sub(r'&?page=\d+', f'&page={page}', base_url)
                elif '?' in base_url:
                    url = f"{base_url}&page={page}"
                else:
                    url = f"{base_url}?page={page}"
            
            print(f"📄 Страница {page}: {url}")
            
            html = self.get_rendered_html(url)
            if html:
                items = self.parse_page(html)
                self.data.extend(items)
                print(f"   ✅ Добавлено: {len(items)}")
                print(f"   📊 Всего: {len(self.data)}\n")
                
                if page < pages:
                    sleep = random.uniform(5, 10)  # Увеличенная пауза
                    print(f"   ⏳ Пауза {sleep:.1f} сек...\n")
                    time.sleep(sleep)
            else:
                print(f"   ❌ Не удалось получить страницу {page}\n")
                break
        
        return self.data
    
    def save_to_csv(self, filename='yandex_realty.csv'):
        """Сохранение в CSV"""
        if not self.data:
            print("\n⚠️ Нет данных для сохранения")
            return None
        
        df = pd.DataFrame(self.data)
        
        # Удобный порядок колонок
        cols = [
            'offer_id', 'price', 'price_numeric', 'old_price', 'area', 'rooms', 
            'floor', 'price_per_m2', 'metro', 'metro_time', 'address', 'author',
            'main_image', 'photo_count', 'badges', 'publish_date', 'url'
        ]
        existing = [c for c in cols if c in df.columns]
        other = [c for c in df.columns if c not in cols]
        df = df[existing + other]
        
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Сохранено {len(self.data)} записей в {filename}")
        return df


# ===================== ЗАПУСК =====================
if __name__ == '__main__':
    # Ваша ссылка с фильтрами
    URL = "https://realty.yandex.ru/moskva_i_moskovskaya_oblast/kupit/kvartira/?price=5000000-20000000&rooms=1-3"
    
    parser = YandexRealtyParser(headless=True)
    parser.start()
    
    try:
        # Начинаем с 1-2 страниц для теста
        parser.parse_multiple_pages(URL, pages=2)
        parser.save_to_csv('yandex_realty_test.csv')
        
        # Показать пример
        if parser.data:
            print("\n📋 Пример первого объявления:")
            for k, v in parser.data[0].items():
                print(f"  {k}: {v}")
    finally:
        parser.close()

In [ ]:
import pandas as pd
df = pd.read_csv('yandex_realty.csv')

In [ ]:
df

In [ ]:
# Count how many rows are duplicates (excluding the first occurrences by default)
count_duplicates = df.duplicated().sum()
print(f"Number of duplicates: {count_duplicates}")

# Count all occurrences of duplicates (by setting keep=False)
count_all_duplicates = df.duplicated(keep=False).sum()
print(f"Total rows involved in duplication: {count_all_duplicates}")


In [ ]:
df['price_numeric'].mean()

In [ ]:
df['metro'].unique()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.hist(df['area'],)

plt.show()

In [ ]:
import requests
import os

for id, row in df.iloc[:1000,:].iterrows():
    image_url = row['main_image']

    folder_path = "cian_images"

    # 2. Создаем папки, если их нет (exist_ok=True не выдаст ошибку, если они уже есть)
    os.makedirs(folder_path, exist_ok=True)

    # 3. Сохраняем файл
    resp = requests.get(image_url)
    file_path = os.path.join(folder_path, f"photo_{id}.jpg")
    if id%50==0:
        print(f"ID: {id}", image_url)
    with open(file_path, "wb") as f:
        f.write(resp.content)